# IS 455 — Showcase: documentation + one “amazing” example pipeline

This notebook **in one place** what your team spread across README / roadmap / backlog files, and walks through **one full pipeline** end-to-end with **example prose** you can imitate.

**Demo pipeline:** `education_records` (registry key) — *Education outcome progress* from `pipeline_backlog.md` §7.

**How to use:** Run all cells top to bottom with working directory **`ml-pipelines/`**.

---
## Part A — What the `ml-pipelines/` folder is for (from `README.MD`)

Place **all course ML pipeline notebooks** here so graders and teammates find them in one folder. Each notebook should be **self-contained**: problem → data → exploration → models → evaluation → causal discussion → (optional) deployment notes.

**Supporting files:**
- `pipeline_kit.py` — shared dual-track modeling and optional `artifacts/` export  
- `blank_notebook.ipynb` — template; set `PIPELINE_NAME`  
- `run_all_pipelines.ipynb` — refresh every registered pipeline  
- `ALL_PIPELINES_ROADMAP.md` — full Tier A/B table and definition of done  
- `pipeline_backlog.md` — business questions for each problem area

---
## Part B — Rubric map (from `ALL_PIPELINES_ROADMAP.md`)

| Section | Turn in |
|--------|---------|
| **1. Problem framing** | Business question, stakeholder, predictive vs explanatory goals, metrics |
| **2. Data, prep, exploration** | Sources, cleaning, joins, exploration evidence |
| **3. Modeling & feature selection** | Interpretable model + predictive model; justify features |
| **4. Evaluation** | Valid split/CV, metrics, **business** interpretation, FP/FN |
| **5. Causal / relationships** | Drivers, correlation vs causation, limits, recommendations |
| **6. Deployment** *(if required)* | How outputs reach users or ops |

**Engine vs author:** `pipeline_kit` trains models and tables; **you** write the narrative.

---
## Part C — Backlog item for this example (`pipeline_backlog.md` §7)

**Education outcome progress**
- **Business question:** Which residents are likely to fall behind in education outcomes?
- **Predictive target:** progress / completion risk (operational triage).
- **Explanatory target:** attendance, level, enrollment signals that explain outcomes.

**Registry key:** `education_records` → dataset `datasets/education_records.csv`.

---
## 1. Problem framing — **example prose** (edit for your voice)

**Business question:** Program leadership needs to know which education records are associated with **non-completion** or struggle so coordinators can prioritize tutoring and check-ins before grades slip.

**Stakeholder:** Education coordinator and case managers assigned to residents.

**Predictive goal:** From enrollment and progress features, **classify** `completion_status` on held-out records so the team can flag cases that may need intervention.

**Explanatory goal:** Use an **interpretable** linear model to describe which factors (e.g. attendance rate, progress percent, level) are most associated with completion patterns—**association**, not proof of causation.

**Success metrics:** Weighted F1 and balanced accuracy for prediction; coefficient stability and plausibility for explanation.

**Why it matters:** Education stability is a core reintegration lever; early identification protects time and trust with the girls we serve.

---
## 2. Data, preparation & exploration — **example prose**

We load `education_records.csv` from the repo `datasets/` folder. Each row is a resident education record with level, enrollment, attendance, and progress fields. Identifiers such as `education_record_id` are excluded from modeling in `pipeline_kit.prepare_education_records`. Exploration should confirm missingness in attendance/progress and the distribution of `completion_status` (class balance).

*Below: quick exploration you can extend with plots.*

In [ ]:
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline_kit import datasets_dir

edu = pd.read_csv(datasets_dir() / "education_records.csv")
print("Shape:", edu.shape)
display(edu.head(6))
print("\nMissing share (top):")
display(edu.isna().mean().sort_values(ascending=False).head(10))
print("\ncompletion_status counts:")
print(edu["completion_status"].value_counts())

---
## 3. Modeling & feature selection — **example prose**

- **Explanatory:** Multinomial logistic regression on scaled numeric + one-hot categorical features (interpretable coefficients).
- **Predictive:** Random forest for stronger out-of-sample separation.
- **Selection:** `pipeline_kit` uses all non-ID features prepared in `prepare_education_records`; for your course write-up, argue whether `resident_id` should be dropped as a proxy for individual rather than program effect (you can refine in `pipeline_kit.py`).

In [ ]:
from pipeline_kit import NotebookConfig, run_notebook_driver

SHOWCASE_PIPELINE = "education_records"

result = run_notebook_driver(
    NotebookConfig(pipeline_name=SHOWCASE_PIPELINE, seed=42, save_artifacts=True)
)

print("Task:", result["task"])
print("Metrics:", result["metrics"])

In [ ]:
from IPython.display import display

print("Explanatory model — top |coefficient| features (association, not causality):")
display(result["explanatory_coefs"])
print("\nPredictive model — random forest importances:")
display(result["predictive_importance"])

---
## 4. Evaluation & interpretation — **example prose**

Read the printed **accuracy**, **balanced accuracy**, and **weighted F1** for both models on the holdout split. In your own submission, **translate** these: e.g. “If we rely on the forest to flag at-risk completion, a false negative means we miss a girl who needed support—a priority error for the mission; a false positive means extra staff time on a case that was actually fine.”

If classes are imbalanced, prefer balanced accuracy and F1 over raw accuracy alone.

---
## 5. Causal and relationship analysis — **example prose**

The tables above rank **associations** between engineered features and `completion_status`. **Do not** claim that changing one input *causes* completion—resident-level unobserved factors and selection into programs can confound results. Use the explanatory model to **generate hypotheses** (e.g. attendance and progress align with completion) and design **monitoring** or **qualitative follow-up**. State data limits (sample size, time coverage) honestly.

---
## 6. Deployment (optional for your course)

Example: export scored CSV weekly from the notebook or a script; case managers sort by predicted risk; retrain when new term data arrives. If your team ships to the web app, point to the artifact schema JSON and where an API would load the trained joblib—only what your instructor requires.

---
## Part D — All Tier A registry keys (copy `PIPELINE_NAME`)

`residents` · `safehouses` · `donation_allocations` · `intervention_plans` · `incident_reports` · `partner_assignments` · `education_records` · `health_wellbeing_records` · `social_media_posts` · `process_recordings` · `home_visitations`

See **`ALL_PIPELINES_ROADMAP.md`** for notebook ↔ CSV mapping and Tier B extras.

In [ ]:
from pipeline_kit import artifact_dir

print("Artifacts for this run (if save_artifacts=True):", artifact_dir())